# Primera entrega — ADAG1 Proyecto Oráculo

**Equipo:** 1  
**Curso:** Análisis y Diseño de Algoritmos · 2026-B  
**Método:** ascenso por coordenadas con dos reinicios reproducibles  
**Modelo:** `Qwen/Qwen3-1.7B` (`qwen17b`)  

## Propósito

Este notebook documenta la búsqueda de una configuración de prompt para el Proyecto Oráculo. La configuración combina cinco ranuras (`rol`, `estrategia`, `formato`, `estilo` y `cierre`) y una temperatura. Cada candidato se evalúa únicamente sobre el conjunto público de `busqueda`; la partición `validacion` se mantiene separada y se consulta una sola vez para medir generalización.

El procedimiento es heurístico: reportamos la **mejor configuración observada**, no una prueba de optimalidad global. El archivo `oraculo.py` es la caja negra del proyecto y no se modifica.

> **Orden de ejecución:** ejecutar las celdas de arriba hacia abajo. La instalación requiere reiniciar la sesión de Colab una vez antes de cargar el modelo. Para reanudar una ejecución, conservar la caché en Google Drive.


## 1. Preparación del entorno

La ejecución completa requiere Google Colab con GPU CUDA, preferiblemente T4. Esta sección monta Google Drive para conservar `cache_oraculo.json`, `registro_busqueda.json` y, cuando se confirme la semana académica, `entrega.json`.


In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

CARPETA_PROYECTO = Path(
    "/content/drive/MyDrive/ADA2026/ProyectoOraculo"
)
CARPETA_PROYECTO.mkdir(parents=True, exist_ok=True)

RUTA_CACHE = CARPETA_PROYECTO / "cache_oraculo.json"
RUTA_REGISTRO = CARPETA_PROYECTO / "registro_busqueda.json"
RUTA_ENTREGA = CARPETA_PROYECTO / "entrega.json"

print("Carpeta persistente:", CARPETA_PROYECTO)
print("Caché:", RUTA_CACHE)
print("Registro:", RUTA_REGISTRO)


In [ ]:
if RUTA_CACHE.exists():
    print("Caché encontrada; las respuestas guardadas se reutilizarán.")
    print("Tamaño:", RUTA_CACHE.stat().st_size, "bytes")
else:
    print("No existe una caché anterior; se creará durante la ejecución.")


In [ ]:
# Instalación idempotente: si open-instruct ya existe, no se clona de nuevo.
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!if [ ! -d /content/open-instruct ]; then git clone -q https://github.com/allenai/open-instruct /content/open-instruct; fi

# El notebook puede ejecutarse desde cualquier carpeta de Colab.
REPO = "https://raw.githubusercontent.com/stokkerdev/ADA-2026-2-GP01/main"
!wget -q --no-cache -O /content/oraculo.py {REPO}/oraculo.py
!wget -q --no-cache -O /content/ayudas.py {REPO}/ayudas.py
!wget -q --no-cache -O /content/datos_visibles.json {REPO}/datos_visibles.json

# Recursos de NLTK usados por los verificadores.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("Instalación terminada.")
print("Reinicia la sesión de Colab y continúa desde la siguiente celda.")


## 2. Carga del modelo, datos y oráculo

Después de reiniciar la sesión de Colab, ejecutar esta celda. Se fija la temperatura en `0.0` para usar decodificación greedy y comparar configuraciones bajo condiciones reproducibles. La semilla controla el reinicio aleatorio de la heurística.


In [ ]:
from pathlib import Path
import json
import random
import time

ROOT = Path("/content")
print("Directorio de trabajo:", ROOT)


In [ ]:
from ayudas import cargar_modelo, cargar_datos, dividir
from oraculo import Oraculo, espacio, RANURAS

MODELO = "qwen17b"
SEMILLA = 1
MAX_PASADAS = 3
REINICIOS = 2
LOTE = 16
PRESUPUESTO = 32_000_000

modelo = cargar_modelo(MODELO)
datos = cargar_datos("/content/datos_visibles.json")
busqueda, validacion = dividir(datos)

# El oráculo se consulta como caja negra; no se modifica su implementación.
oraculo = Oraculo(
    modelo,
    busqueda,
    validacion,
    lote=LOTE,
    presupuesto=PRESUPUESTO,
    cache=str(RUTA_CACHE),
)

print("Configuraciones con temperatura 0.0:", len(espacio()))
print("Instancias de búsqueda:", len(busqueda))
print("Instancias de validación:", len(validacion))


In [ ]:
# Comprobación rápida de que el conjunto de datos tiene las particiones esperadas.
assert len(busqueda) == 100
assert len(validacion) == 87
assert all("id" in x and "ids" in x for x in datos)
print("Comprobación de datos: OK")


## 3. Registro reproducible y función objetivo

Cada evaluación registra la configuración, la precisión observada, el reinicio, la pasada, la ranura modificada, el tiempo y las primeras trazas de fallo. La precisión es:

```text
precision(c) = instancias resueltas completamente / instancias evaluadas
```

La evaluación es todo-o-nada: si una respuesta incumple una sola restricción, esa instancia vale `0`.


In [ ]:
registro = []
cache_resultados = {}
contador_evaluaciones = 0
mejor_precision_global = -1.0

# Cota teórica del procedimiento:
# por reinicio: 1 evaluación inicial + MAX_PASADAS * 5 ranuras * 8 opciones.
MAX_EVALUACIONES = REINICIOS * (1 + MAX_PASADAS * len(RANURAS) * 8)

def clave_config(c):
    return tuple(
        c.get(k)
        for k in ("rol", "estrategia", "formato", "estilo", "cierre", "temperatura")
    )

def evaluar_registrada(config, *, reinicio, pasada, ranura, opcion):
    global contador_evaluaciones, mejor_precision_global

    inicio = time.perf_counter()
    resultado = oraculo.evaluar(config, busqueda, semilla=SEMILLA)
    segundos = time.perf_counter() - inicio
    precision = float(resultado.precision)

    contador_evaluaciones += 1
    mejor_precision_global = max(mejor_precision_global, precision)
    trazas = resultado.trazas or []

    registro.append({
        "evaluacion": contador_evaluaciones,
        "configuracion": dict(config),
        "precision": precision,
        "semilla": SEMILLA,
        "reinicio": reinicio,
        "pasada": pasada,
        "ranura": ranura,
        "opcion": opcion,
        "segundos": segundos,
        "n_instancias": len(busqueda),
        "primeros_fallos": [
            {"id": t.get("id"), "violo": t.get("violo")}
            for t in trazas[:10]
        ],
    })
    cache_resultados[clave_config(config)] = resultado

    print(
        f"eval {contador_evaluaciones:3d}/{MAX_EVALUACIONES}   "
        f"actual {precision:5.1%}   "
        f"mejor {mejor_precision_global:5.1%}",
        flush=True,
    )
    return resultado


## 4. Heurística: ascenso por coordenadas con reinicios

La heurística modifica una sola ranura por vez. Para cada ranura prueba sus ocho opciones, conserva la mejor alternativa y continúa con la siguiente. Una pasada completa recorre `rol → estrategia → formato → estilo → cierre`. Si una pasada no mejora la precisión, ese reinicio termina.

Se usan dos puntos iniciales reproducibles: todos los índices en cero y una configuración generada con `SEMILLA = 1`. El mejor resultado observado entre ambos reinicios se reporta como solución heurística; no se afirma optimalidad global.


In [ ]:
def busqueda_por_coordenadas(inicial, reinicio):
    actual = dict(inicial)
    resultado_inicial = evaluar_registrada(
        actual, reinicio=reinicio, pasada=0, ranura="inicial", opcion=None
    )
    precision_actual = float(resultado_inicial.precision)

    for pasada in range(1, MAX_PASADAS + 1):
        hubo_mejora = False

        for ranura in RANURAS:
            mejor_local = dict(actual)
            precision_local = precision_actual

            for opcion in range(8):
                candidato = dict(actual)
                candidato[ranura] = opcion
                resultado = evaluar_registrada(
                    candidato,
                    reinicio=reinicio,
                    pasada=pasada,
                    ranura=ranura,
                    opcion=opcion,
                )
                precision_candidato = float(resultado.precision)

                if precision_candidato > precision_local:
                    precision_local = precision_candidato
                    mejor_local = candidato

            if precision_local > precision_actual:
                actual = mejor_local
                precision_actual = precision_local
                hubo_mejora = True

        print(
            f"Fin de pasada {pasada}, reinicio {reinicio}: "
            f"{precision_actual:.1%}"
        )
        if not hubo_mejora:
            break

    return actual, precision_actual

# Puntos iniciales reproducibles, sin usar una configuración objetivo conocida.
iniciales = [
    {k: 0 for k in RANURAS} | {"temperatura": 0.0},
]
rng = random.Random(SEMILLA)
iniciales.append(
    {k: rng.randrange(8) for k in RANURAS} | {"temperatura": 0.0}
)

resultados_reinicios = []
for numero, inicial in enumerate(iniciales, start=1):
    configuracion, precision = busqueda_por_coordenadas(inicial, reinicio=numero)
    resultados_reinicios.append({
        "reinicio": numero,
        "configuracion": configuracion,
        "precision": precision,
    })
    print(f"Reinicio {numero}: {precision:.2%} — {configuracion}")

mejor_observado = max(resultados_reinicios, key=lambda x: x["precision"])
MEJOR_CONFIG = mejor_observado["configuracion"]
MEJOR_PRECISION = float(mejor_observado["precision"])

print("\nMejor configuración observada:", MEJOR_CONFIG)
print("Precisión en búsqueda:", f"{MEJOR_PRECISION:.2%}")
print("Evaluaciones realizadas:", contador_evaluaciones)
print("Máximo teórico:", MAX_EVALUACIONES)


## 5. Análisis de trazas

Las trazas contienen la primera restricción que falló en cada instancia no resuelta. El resumen siguiente se agrega sobre las configuraciones evaluadas y sirve para identificar familias difíciles; no representa fallos de una sola configuración ni debe interpretarse como un conteo de instancias únicas.


In [ ]:
from collections import Counter

fallos = Counter()
for fila in registro:
    for traza in fila["primeros_fallos"]:
        if traza.get("violo"):
            fallos[traza["violo"]] += 1

print("Restricciones más frecuentes en las primeras trazas:")
for nombre, cantidad in fallos.most_common(15):
    print(f"{nombre}: {cantidad}")


## 6. Validación final

La configuración ganadora se evalúa una sola vez sobre las 87 instancias reservadas. Esta partición no se utiliza para cambiar la configuración; solo mide generalización.


In [ ]:
r_val = oraculo.validar(MEJOR_CONFIG, n=len(validacion))
VALIDACION_PRECISION = float(r_val.precision)

print("Configuración elegida:", MEJOR_CONFIG)
print("Precisión en búsqueda:", f"{MEJOR_PRECISION:.2%}")
print("Precisión en validación:", f"{VALIDACION_PRECISION:.2%}")


In [ ]:
# Guardar evidencia reproducible en Google Drive.
with RUTA_REGISTRO.open("w", encoding="utf-8") as archivo:
    json.dump(
        {
            "modelo": MODELO,
            "semilla": SEMILLA,
            "temperatura": MEJOR_CONFIG["temperatura"],
            "metodo": "ascenso_por_coordenadas_con_reinicios",
            "max_pasadas": MAX_PASADAS,
            "reinicios": REINICIOS,
            "max_evaluaciones": MAX_EVALUACIONES,
            "evaluaciones_realizadas": contador_evaluaciones,
            "resultados_reinicios": resultados_reinicios,
            "mejor_configuracion": MEJOR_CONFIG,
            "precision_busqueda": MEJOR_PRECISION,
            "precision_validacion": VALIDACION_PRECISION,
            "evaluaciones": registro,
        },
        archivo,
        ensure_ascii=False,
        indent=2,
    )
print("Registro guardado en:")
print(RUTA_REGISTRO)


## 7. Generación de `entrega.json`

Ejecutar la siguiente celda únicamente cuando se haya confirmado la semana académica oficial. El valor `0` no debe entregarse como semana si la profesora o el cronograma indican otro número.


In [ ]:
from ayudas import entrega

GRUPO = "G01"
SEMANA_ACADEMICA = None  # Completar con el número oficial antes de entregar.

if SEMANA_ACADEMICA is None:
    print("Pendiente: confirmar SEMANA_ACADEMICA antes de generar entrega.json")
else:
    entrega(
        GRUPO,
        MEJOR_CONFIG,
        SEMANA_ACADEMICA,
        ruta=str(RUTA_ENTREGA),
    )
    print("Entrega guardada en:")
    print(RUTA_ENTREGA)


## 8. Resultados registrados y conclusión

La ejecución experimental registrada produjo:

| Resultado | Valor |
|---|---:|
| Evaluaciones de configuraciones | 162 de 242 máximas |
| Precisión en búsqueda | 16.00% (16/100) |
| Precisión en validación | 10.34% (9/87) |

La mejor configuración observada fue:

```python
{
    "rol": 1,
    "estrategia": 4,
    "formato": 0,
    "estilo": 5,
    "cierre": 0,
    "temperatura": 0.0,
}
```

La diferencia entre búsqueda y validación es una limitación esperable de la búsqueda heurística y se reporta sin reajustar la configuración con los datos reservados. El resultado es la mejor configuración encontrada bajo el modelo, semilla, temperatura, presupuesto y particiones utilizados; no constituye una prueba de óptimo global.
